# Jupyter Notebook to help manage calculations

In [1]:
import os
import subprocess
import sys
import arkane

DFT_DIR = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'dft')
sys.path.append(DFT_DIR)
import autotst_wrapper



/home/harris.se/rmg/RMG-Py/rmgpy/rmg/reactors.py:53: RuntimeWarning: Unable to import Julia dependencies, original error: [Errno 2] No such file or directory: 'julia': 'julia'
  warnings.warn("Unable to import Julia dependencies, original error: " + str(e), RuntimeWarning)


Loading DFT database from /work/westgroup/harris.se/autoscience/reaction_calculator/database


# What do we need to calculate?

In [7]:
# Enter the species you are supposed to calculate (using database indices)
# species_to_calculate = [4, 99]
# species_to_calculate = [i for i in range(150)]


# Round 1
species_to_calculate = [4, 57, 280, 60, 61, 73, 21, 72, 32]
reactions_to_calculate = [4721]
assert len(species_to_calculate) + len(reactions_to_calculate) == 10

# reactions_to_calculate = [
#     288, 4724, 5046, 4778, 4736, 4729, 4728, 5047,
#     4779, 286, 246, 5596, 808, 915, 4737, 5446,
#     324, 4738, 7841, 804, 809, 4721, 945, 213, 289,
#     422, 805, 4796, 1077, 1111, 1706, 4917, 417, 319,
#     313, 278, 314, 52, 5056, 405, 5102, 521, 404, 410,
#     4733, 296, 321, 301, 280, 253, 459, 1736, 1778
# ]
# reactions_to_calculate = [
#     288, 4724, 5046, 4778, 4736, 4729, 4728, 50, 4752, 5047,
#     4779, 286, 246, 5596, 808, 915, 4732, 518, 4737, 5446,
#     324, 4738, 7841, 804, 809, 4721, 245, 945, 213, 289,
#     422, 805, 4796, 1077, 1111, 1706, 4917, 417, 319,
#     313, 278, 314, 52, 5056, 405, 5102, 521, 404, 410,
#     4733, 296, 321, 301, 280, 253, 459, 1736, 1778, 299
# ]

# problem_reactions = [50, 4752, 4732, 518, 245, 299]


# Check which things are complete

In [8]:
def run_command(command):
    p = subprocess.Popen(
        command,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    output = p.communicate()
    text = output[0].decode('utf-8')
    text_list = text.split('\n')
    if text_list[-1] == '':
        text_list = text_list[:-1]
    return text_list

In [14]:
# make sure 'dlpno' AND 'orca terminated normally' are in the logfile that was used
DFT_DIR = os.environ['DFT_DIR']

finished_sp_single_points = run_command('grep -il "orca terminated normally" '+ os.path.join(DFT_DIR, 'thermo/species_*/single_point/conformer.out'))
finished_ts_single_points = run_command('grep -il "orca terminated normally" '+ os.path.join(DFT_DIR, 'kinetics/reaction_*/single_point/conformer.out'))


finished_arkane_thermo = run_command('grep -il "dlpno" '+ os.path.join(DFT_DIR, 'thermo/species_*/arkane/RMG_libraries/thermo.py'))
finished_arkane_kinetics = run_command('grep -il "dlpno" '+ os.path.join(DFT_DIR, 'kinetics/reaction_*/arkane/RMG_libraries/reactions.py'))

# Go through species

In [15]:
unfinished_species = []
for i in species_to_calculate:
    orca_logname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/single_point/conformer.out')
    arkane_fname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/arkane/RMG_libraries/thermo.py')
    if orca_logname not in finished_sp_single_points or arkane_fname not in finished_arkane_thermo:
        print(f'Species {i} not done yet')
        unfinished_species.append(i)
if len(unfinished_species) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_species)} left to calculate')

DONE!


# Go through reactions

In [16]:
unfinished_reactions = []
for i in reactions_to_calculate:
    orca_logname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/single_point/conformer.out')
    arkane_fname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/arkane/RMG_libraries/reactions.py')
    if orca_logname not in finished_ts_single_points or arkane_fname not in finished_arkane_kinetics:
        print(f'Reaction {i} not done yet')
        unfinished_reactions.append(i)

if len(unfinished_reactions) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_reactions)} left to calculate')

Reaction 4721 not done yet

1 left to calculate


# Go through each stage to see what needs calculating

## 1. Geometry Optimization

In [17]:
unfinished_opt = []
for i in unfinished_species:
    conformer_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'conformers')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=1.0)
    if not geo_opt_completed:
        print(f'Species {i} optimization not done yet')
        unfinished_opt.append(i)

if len(unfinished_opt) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_opt)} left to calculate')

DONE!


In [ ]:
# option to set off species optimizations
for i in unfinished_opt:
    print(f'Starting species conformer optimization {i}')
    autotst_wrapper.setup_opt(i, 'shell')
    autotst_wrapper.run_opt(i, 'shell')

## 2. Rotor scans

In [18]:
unfinished_rotors = []
for i in unfinished_species:
    rotor_dir = os.path.join(DFT_DIR, 'thermo', f'species_{i:04}', 'rotors')
    rotors = not os.path.exists(os.path.join(rotor_dir, 'NO_ROTORS.txt'))
    if rotors:
        rotors_completed = autotst_wrapper.conformers_done_optimizing(rotor_dir, completion_threshold=1.0, base_name='rotor_')
        if not rotors_completed:
            print(f'Species {i} rotors not done yet')
            unfinished_rotors.append(i)
            print()
        
if len(unfinished_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rotors)} left to calculate')

DONE!


In [ ]:
# option to set off species
for i in unfinished_rotors:
    print(f'Starting species rotor scans {i}')
    autotst_wrapper.setup_rotors(i)
    autotst_wrapper.run_rotors(i)

## 3. DLPNO-CCSDT(T)-F12 Single Point Calculations

In [11]:
unfinished_single_points = []
for i in unfinished_species:
    orca_logname = os.path.join(DFT_DIR, f'thermo/species_{i:04}/single_point/conformer.out')
    if orca_logname not in finished_sp_single_points:
        print(f'Species {i} single point calculation not done yet')
        unfinished_single_points.append(i)
        print()
        
if len(unfinished_single_points) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_single_points)} left to calculate')

DONE!


In [ ]:
# TODO run single points
subset = [unfinished_single_points[0]]
assert len(subset) <= 5
for i in subset:
    print(f'Starting species single point calc {i}')
    autotst_wrapper.setup_single_point(i, calc_type='species', force_rerun=True, parallel=True)
    autotst_wrapper.run_single_point(i, calc_type='species', force_rerun=True)


## 4. Run Arkane

In [12]:
unfinished_species_arkanes = []
for i in unfinished_species:
    arkane_result = os.path.join(DFT_DIR, f'thermo/species_{i:04}/arkane/RMG_libraries/thermo.py')
    if arkane_result not in finished_arkane_thermo:
        print(f'Species {i} Arkane not done yet')
        unfinished_species_arkanes.append(i)
        print()
        
if len(unfinished_species_arkanes) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_species_arkanes)} left to calculate')

Species 280 Arkane not done yet


1 left to calculate


In [13]:
# option to set off species optimizations
subset = [280]
assert len(subset) < 5
for i in subset:
    print(f'Starting species arkane run {i}')
    autotst_wrapper.setup_arkane_species(i, force_rerun=True)
    autotst_wrapper.run_arkane_species(i, force_rerun=True)


Starting species arkane run 280
2024-12-26 18:05:52.559037 Forcing rerun of Arkane species setup
2024-12-26 18:05:52.567988 Setting up Arkane species with rotors=True
2024-12-26 18:05:54.198976 Forcing rerun of arkane species
Submitted batch job 45856238



# Reaction Calculation Steps

## 1. Shell Optimization (reaction center frozen)

In [ ]:
unfinished_shell = []
for i in unfinished_reactions:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'shell')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.9, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} shell optimization not done yet')
        unfinished_shell.append(i)

if len(unfinished_shell) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_shell)} left to calculate')

In [ ]:
for i in unfinished_shell:
    print(f'Starting reaction shell optimization {i}')
    autotst_wrapper.setup_opt(i, 'shell')
    autotst_wrapper.run_opt(i, 'shell')

## 2. Center Optimization (shell frozen - optimize reaction center to loose TS)

In [ ]:
unfinished_center = []
for i in unfinished_reactions:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'center')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.9, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} center optimization not done yet')
        unfinished_center.append(i)

if len(unfinished_center) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_center)} left to calculate')

In [ ]:
for i in unfinished_center:
    print(f'Starting reaction center optimization {i}')
    autotst_wrapper.setup_opt(i, 'center')
    autotst_wrapper.run_opt(i, 'center')

## 3. Overall TS Optimization (no constraints)

In [19]:
unfinished_overall = []
for i in unfinished_reactions:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'overall')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.9, base_name='fwd_ts_')
    if not geo_opt_completed:
        print(f'Reaction {i} overall optimization not done yet')
        unfinished_overall.append(i)

if len(unfinished_overall) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_overall)} left to calculate')

DONE!


In [ ]:
for i in unfinished_overall:
    print(f'Starting reaction overall optimization {i}')
    autotst_wrapper.setup_opt(i, 'overall')
    autotst_wrapper.run_opt(i, 'overall')

## 4. Reaction TS Rotor Scans

In [20]:
unfinished_ts_rotors = []
for i in unfinished_reactions:
    conformer_dir = os.path.join(DFT_DIR, 'kinetics', f'reaction_{i:06}', 'rotors')
    geo_opt_completed = autotst_wrapper.conformers_done_optimizing(conformer_dir, completion_threshold=0.9, base_name='rotor_')
    if not geo_opt_completed:
        print(f'Reaction {i} TS rotors not done yet')
        unfinished_ts_rotors.append(i)

if len(unfinished_ts_rotors) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_ts_rotors)} left to calculate')

No conformers with glob string /work/westgroup/harris.se/autoscience/reaction_calculator/dft/kinetics/reaction_004721/rotors/rotor_*.com
Reaction 4721 TS rotors not done yet

1 left to calculate


In [ ]:
subset = unfinished_ts_rotors[1:5]

In [ ]:
subset

In [ ]:
subset = unfinished_ts_rotors[0:5]
assert len(subset) <= 5

for i in subset:
    print(f'Starting reaction TS rotors {i}')
    autotst_wrapper.setup_ts_rotors(i, force_rerun=True)
    autotst_wrapper.run_ts_rotors(i, force_rerun=True)

## 5. Reaction Single Point Calculations

In [21]:
unfinished_rxn_single_points = []
for i in unfinished_reactions:
    orca_logname = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/single_point/conformer.out')
    if orca_logname not in finished_ts_single_points:
        print(f'Reaction {i} single point not done yet')
        unfinished_rxn_single_points.append(i)

if len(unfinished_rxn_single_points) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rxn_single_points)} left to calculate')

DONE!


In [ ]:
for i in unfinished_rxn_single_points:
    print(f'Starting reaction single point {i}')
    autotst_wrapper.setup_single_point(i, calc_type='reaction', force_rerun=True, parallel=True)
    autotst_wrapper.run_single_point(i, calc_type='reaction', force_rerun=True)

#     try:
#         autotst_wrapper.setup_single_point(i, calc_type='reaction', force_rerun=True, parallel=False)
#         autotst_wrapper.run_single_point(i, calc_type='reaction', force_rerun=True)
#     except (TypeError, arkane.exceptions.LogError):
#         pass

## 6. Reaction Arkane

In [ ]:
unfinished_rxn_arkane = []
for i in unfinished_reactions:
    arkane_result = os.path.join(DFT_DIR, f'kinetics/reaction_{i:06}/arkane/RMG_libraries/reactions.py')
    if arkane_result not in finished_arkane_kinetics:
        print(f'Reaction {i} Arkane not done yet')
        unfinished_rxn_arkane.append(i)

if len(unfinished_rxn_arkane) == 0:
    print('DONE!')
else:
    print(f'\n{len(unfinished_rxn_arkane)} left to calculate')

In [ ]:
for i in unfinished_rxn_arkane:
    print(f'Starting reaction Arkane {i}')
    autotst_wrapper.setup_arkane_reaction(i)
    autotst_wrapper.run_arkane_reaction(i)